<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/lesson6/Lesson6_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 6 — Exercises**
## Eigenstructure and Stability: What Happens When a Matrix Is Applied Again and Again

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 2 — Linear Algebra II\
**Lesson:** Lesson 6

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | Eigenvalues and eigenvectors by hand | 10 |
| 2 | Types of eigenvalues and rotation | 15 |
| 3 | Diagonalisation, modes and power iteration | 15 |
| 4 | Stability and timescales of a recurrent network | 15 |
| 5 | Continuous time: a rate network and an integrator | 15 |
| 6 | Symmetric matrices | 15 |
| 7 | Real data: is the timescale in the brain or in my smoothing? | 15 |
| **Total** | | **100** |
| Bonus | The exponential of a matrix | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(7)   # exercise notebook uses its own fixed seed
PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

def rotation(deg):
    t = np.radians(deg)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

def iterate(W, x0, n_steps):
    # Trajectory x_0, ..., x_n of x_{t+1} = W x_t, shape (n_steps + 1, len(x0)).
    xs = [np.asarray(x0, float)]
    for _ in range(n_steps):
        xs.append(W @ xs[-1])
    return np.array(xs)

print("Setup complete.")

---
## Exercise 1 — Eigenvalues and Eigenvectors by Hand (10 points)

$$\mathbf{M} = \begin{pmatrix}4 & 1\\ 2 & 3\end{pmatrix}$$

**1.1 (4 points, written response, no code).** Write the characteristic equation of $\mathbf{M}$ using its trace and determinant, and solve it. For each eigenvalue, find an eigenvector by solving $(\mathbf{M} - \lambda\mathbf{I})\mathbf{v} = \mathbf{0}$. Check that the eigenvalues add up to the trace and multiply to the determinant.

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 1.2 (3 points): Check your answers in code.
#   lam_M, V_M -- from np.linalg.eig(M) (eigenvalues, and eigenvectors as COLUMNS)
#   checks     -- a list of two booleans: np.allclose(M v, lambda v) for each eigenpair

M = np.array([[4.0, 1.0], [2.0, 3.0]])

# YOUR CODE HERE
lam_M, V_M = None, None
checks = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert lam_M is not None and V_M is not None and checks is not None, "lam_M, V_M and checks are still None."
assert np.shape(V_M) == (2, 2) and len(checks) == 2 and all(checks), "Both eigenpairs should satisfy M v = lambda v."
assert np.isclose(np.sum(lam_M), np.trace(M)) and np.isclose(np.prod(lam_M), np.linalg.det(M)), "Eigenvalues should sum to the trace and multiply to the determinant."
print("Self-check passed.")

**1.3 (3 points, written response).** NumPy's eigenvectors are scaled to length 1, so they look different from yours. Explain why both are correct. If $\mathbf{M}$ were the weight matrix of a two-neuron network, $\mathbf{x}_{t+1} = \mathbf{M}\mathbf{x}_t$, describe in words what happens to activity that starts along $(1, 1)$ and along $(1, -2)$.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — Types of Eigenvalues and Rotation (15 points)

In [ ]:
# 2.1 (5 points): Write eigen_type(A) for a 2x2 matrix that returns 'real distinct', 'repeated' or 'complex'
# from the discriminant tr(A)^2 - 4 det(A) (treat |disc| < 1e-10 as zero).
# Then classify every matrix in `mats`, storing the results in the dict `types`.

mats = {
    'P': np.array([[3.0, 2.0], [1.0, 4.0]]),
    'Q': np.array([[0.0, -1.0], [1.0, 0.0]]),
    'R': np.array([[2.0, 1.0], [0.0, 2.0]]),
    'S': np.array([[1.0, -2.0], [3.0, 1.0]]),
    'T': np.array([[5.0, 0.0], [0.0, 5.0]]),
}

def eigen_type(A):
    # YOUR CODE HERE
    raise NotImplementedError

types = None   # replace with a dict {name: eigen_type(matrix)}

In [ ]:
# SELF-CHECK 2.1 -- tested on matrices not in `mats`.
assert types is not None and set(types) == set(mats), "types needs one entry per matrix."
assert eigen_type(np.diag([1.0, 2.0])) == 'real distinct', "eigen_type: diag(1, 2) has two different real eigenvalues."
assert eigen_type(rotation(45)) == 'complex', "eigen_type: a rotation has complex eigenvalues."
assert eigen_type(np.array([[3.0, 5.0], [0.0, 3.0]])) == 'repeated', "eigen_type: [[3, 5], [0, 3]] has a repeated eigenvalue."
print("Self-check passed.")

In [ ]:
# 2.2 (5 points): A rotating, decaying network. Build W22 = 0.9 * rotation(30).
#   lam22    -- its eigenvalues
#   modulus  -- |lambda| (same for both)
#   angle    -- |arg(lambda)| in degrees (np.angle gives radians)
#   traj22   -- iterate(W22, [1, 0], 30)
# Plot the trajectory in the (neuron 1, neuron 2) plane, equal axes, labels and a title giving |lambda| and the angle.

# YOUR CODE HERE
W22 = None
lam22 = None
modulus = None
angle = None
traj22 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert W22 is not None and lam22 is not None and modulus is not None and angle is not None and traj22 is not None, "Fill in all five results."
assert np.shape(traj22) == (31, 2), "traj22 should hold 31 states of 2 neurons."
assert np.allclose(np.linalg.norm(traj22[1:], axis=1) / np.linalg.norm(traj22[:-1], axis=1), modulus), "Each step should shrink the state by |lambda|."
print("Self-check passed.")

**2.3 (5 points, written response).** (a) Which matrices in 2.1 are complex, and what do their eigenvalues tell you about repeated application? (b) $\mathbf{R}$ and $\mathbf{T}$ both have a repeated eigenvalue. Which one has only a single eigenvector direction, and how can you tell? (c) In 2.2, how many steps does one full turn take, and after one turn, by what factor has the activity shrunk?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — Diagonalisation, Modes and Power Iteration (15 points)

In [ ]:
W3 = np.array([[0.70, 0.20, 0.00],
               [0.10, 0.60, 0.20],
               [0.05, 0.10, 0.50]])      # a three-neuron recurrent network
x0_3 = np.array([1.0, -1.0, 2.0])

In [ ]:
# 3.1 (5 points): Diagonalise W3.
#   lam3, V3 -- eigenvalues and eigenvectors, SORTED by decreasing |lambda| (use np.argsort on -abs(lam))
#   recon_ok -- True if V3 diag(lam3) V3^{-1} equals W3
#   power_ok -- True if V3 diag(lam3**25) V3^{-1} equals np.linalg.matrix_power(W3, 25)

# YOUR CODE HERE
lam3, V3 = None, None
recon_ok = None
power_ok = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1
assert lam3 is not None and V3 is not None and recon_ok is not None and power_ok is not None, "Fill in lam3, V3, recon_ok and power_ok."
assert np.all(np.diff(np.abs(lam3)) <= 1e-12), "Sort the eigenvalues by decreasing |lambda|."
assert np.allclose(W3 @ V3, V3 * lam3), "Column k of V3 should be an eigenvector for lam3[k]."
print("Self-check passed.")

In [ ]:
# 3.2 (5 points): Mode decomposition of x0_3.
#   c3       -- coordinates of x0_3 in the eigenvector basis (solve V3 c = x0_3)
#   x_modes  -- the state at t = 10 built from the modes: sum_k c3[k] * lam3[k]**10 * V3[:, k]   (take .real)
#   x_direct -- the state at t = 10 from iterate(W3, x0_3, 10)[-1]
#   align    -- |cosine similarity| between x_direct and the dominant eigenvector V3[:, 0] (real part)

# YOUR CODE HERE
c3 = None
x_modes = None
x_direct = None
align = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert c3 is not None and x_modes is not None and x_direct is not None and align is not None, "Fill in c3, x_modes, x_direct and align."
assert np.allclose(V3 @ c3, x0_3), "c3 should satisfy V3 c3 = x0_3."
assert np.allclose(x_modes, x_direct), "The mode sum should equal the directly iterated state."
assert 0 <= align <= 1 + 1e-12, "align is an absolute cosine similarity."
print("Self-check passed.")

In [ ]:
# 3.3 (5 points): Power iteration. Write power_iteration(A, n_iter) that starts from a fixed vector of ones,
# repeatedly multiplies by A and rescales to length 1, and returns (v, lam_est) with lam_est = v . (A v).
# Record the error |lam_est - lam3[0]| after n_iter = 1, 2, 5, 10, 20, 40 in the array power_err.

def power_iteration(A, n_iter):
    # YOUR CODE HERE
    raise NotImplementedError

power_err = None   # replace

In [ ]:
# SELF-CHECK 3.3 -- tested on a matrix with known dominant eigenvalue.
v_chk, l_chk = power_iteration(np.diag([3.0, 1.0, 0.5]) + 0.01, 60)
assert abs(l_chk - np.abs(np.linalg.eigvals(np.diag([3.0, 1.0, 0.5]) + 0.01)).max()) < 1e-6, "power_iteration should converge to the dominant eigenvalue."
assert power_err is not None and np.shape(power_err) == (6,), "power_err needs one value per iteration count."
print("Self-check passed.")

---
## Exercise 4 — Stability and Timescales of a Recurrent Network (15 points)

A network of 60 neurons has random connectivity $\mathbf{W}_0$ and an overall gain $g$: $\mathbf{x}_{t+1} = g\,\mathbf{W}_0\mathbf{x}_t$, with one time step $\Delta t = 10$ ms.

In [ ]:
N4 = 60
W0_4 = RNG.normal(0, 1 / np.sqrt(N4), size=(N4, N4))
dt4 = 0.01                                   # seconds per step
x0_4 = RNG.normal(size=N4)

In [ ]:
# 4.1 (5 points):
#   rho0   -- the spectral radius of W0_4
#   g_crit -- the gain at which g * W0_4 has spectral radius exactly 1
#   For g in [0.5 * g_crit, g_crit, 1.2 * g_crit], simulate 200 steps from x0_4 and store ||x_200|| / ||x_0|| in the list growth.

# YOUR CODE HERE
rho0 = None
g_crit = None
growth = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1
assert rho0 is not None and g_crit is not None and growth is not None, "Fill in rho0, g_crit and growth."
assert np.isclose(np.abs(np.linalg.eigvals(g_crit * W0_4)).max(), 1.0), "At g_crit the spectral radius should be exactly 1."
assert len(growth) == 3 and growth[0] < 1e-3 and growth[2] > 1e3, "Below the critical gain activity should die away; above it, it should explode."
print("Self-check passed.")

In [ ]:
# 4.2 (5 points): Timescales at g = 0.9 * g_crit.
#   lam42  -- eigenvalues of 0.9 * g_crit * W0_4
#   taus   -- decay time constants -dt4 / ln|lambda| (seconds), one per eigenvalue
#   freqs  -- frequencies |arg lambda| / (2 pi dt4) (Hz), one per eigenvalue
# Plot taus against freqs (scatter), log y-axis, labels, title.

# YOUR CODE HERE
lam42 = None
taus = None
freqs = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert lam42 is not None and taus is not None and freqs is not None, "Fill in lam42, taus and freqs."
assert np.shape(taus) == np.shape(freqs) == (N4,), "One time constant and one frequency per eigenvalue."
assert np.all(taus > 0), "At 0.9 x critical gain every mode decays, so every tau is positive."
assert np.isclose(taus.max(), -dt4 / np.log(0.9)), "The slowest mode has |lambda| = 0.9 exactly here."
print("Self-check passed.")

**4.3 (5 points, written response).** Explain the three values in `growth` using the spectral radius. In 4.2, why is the slowest time constant exactly $-\Delta t/\ln 0.9$? A single model neuron here "remembers" for one step (10 ms). What does the network do for its memory, and what happens to the slowest timescale as $g \to g_{\text{crit}}$?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Continuous Time: a Rate Network and an Integrator (15 points)

A linear rate network obeys $\tau_n\,d\mathbf{x}/dt = -\mathbf{x} + \mathbf{W}\mathbf{x} + \mathbf{u}(t)$ with $\tau_n = 20$ ms.

In [ ]:
tau5 = 0.02
W5 = np.array([[0.6, 0.3], [0.3, 0.6]])      # symmetric two-neuron network

In [ ]:
# 5.1 (5 points):
#   mu5     -- eigenvalues of W5 (use np.linalg.eigvalsh, since W5 is symmetric)
#   A5      -- the continuous-time matrix (W5 - I) / tau5
#   lamA5   -- eigenvalues of A5 (eigvalsh)
#   tau_modes -- the two mode time constants -1 / lamA5 (seconds)
#   stable5 -- True if every eigenvalue of A5 has negative real part

# YOUR CODE HERE
mu5 = None
A5 = None
lamA5 = None
tau_modes = None
stable5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.1
assert mu5 is not None and A5 is not None and lamA5 is not None and tau_modes is not None and stable5 is not None, "Fill in all five results."
assert np.allclose(np.sort(lamA5), np.sort((mu5 - 1) / tau5)), "Each eigenvalue of A should be (mu - 1) / tau for an eigenvalue mu of W."
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): Euler simulation versus the exact solution. With NO input, start from x0 = (1, 0) and simulate
#   x <- x + dt * A5 x   for 0.3 s with dt = 1 ms,
# storing the trajectory in x_euler (shape (301, 2), including the start). The exact solution is
#   x(t) = V diag(exp(lambda t)) V^T x0   (V from np.linalg.eigh(A5); V^T = V^{-1} because A5 is symmetric).
# Store it in x_exact (same shape) and the largest absolute difference in max_diff. Plot both for neuron 1.

dt5 = 0.001
t5 = np.arange(0, 0.3 + dt5 / 2, dt5)

# YOUR CODE HERE
x_euler = None
x_exact = None
max_diff = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert x_euler is not None and x_exact is not None and max_diff is not None, "Fill in x_euler, x_exact and max_diff."
assert np.shape(x_euler) == np.shape(x_exact) == (301, 2), "Both trajectories should have shape (301, 2)."
assert np.allclose(x_exact[0], [1, 0]), "The exact solution should start at x0."
assert max_diff < 0.05, "With dt = 1 ms, Euler should track the exact solution closely."
print("Self-check passed.")

In [ ]:
# 5.3 (5 points): Tune W5 into an integrator. Find the value w_int of the off-diagonal weight (keeping the diagonal
# at 0.6) that gives W an eigenvalue exactly 1. Then, starting from x = 0, apply a 50 ms input pulse u = (1, 1)
# (from t = 50 ms to t = 100 ms) and simulate 1 s with Euler (dt = 1 ms). Store neuron 1's rate in held (shape (1001,)).
# Hint: the eigenvalues of [[a, b], [b, a]] are a + b and a - b.

t53 = np.arange(0, 1.0 + 0.0005, 0.001)

# YOUR CODE HERE
w_int = None
held = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.3
assert w_int is not None and held is not None, "w_int and held are still None."
assert np.isclose(np.linalg.eigvalsh(np.array([[0.6, w_int], [w_int, 0.6]])).max(), 1.0), "With w_int the largest eigenvalue of W should be exactly 1."
assert np.shape(held) == (1001,), "held should have one value per millisecond from 0 to 1 s."
assert abs(held[-1] - held[200]) < 0.01 * abs(held[200]), "After the pulse the integrator should hold its value."
print("Self-check passed.")

---
## Exercise 6 — Symmetric Matrices (15 points)

Symmetric matrices, such as the symmetric connectivity of Exercise 5 or the covariance matrices of Lesson 7, have real eigenvalues and orthogonal eigenvectors. Here you check these facts numerically and see why the eigenvectors matter for symmetric matrices in particular.

In [ ]:
# 6.1 (5 points): Build a random symmetric 5x5 matrix S6 = (B + B^T) / 2 from B = RNG.normal(size=(5, 5)).
#   lam6, Q6 -- from np.linalg.eigh(S6)
#   max_imag -- the largest |imaginary part| among np.linalg.eigvals(S6) (should be ~0)
#   ortho_ok -- True if Q6^T Q6 equals the identity
#   recon6_ok -- True if Q6 diag(lam6) Q6^T equals S6

# YOUR CODE HERE
S6 = None
lam6, Q6 = None, None
max_imag = None
ortho_ok = None
recon6_ok = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1
assert S6 is not None and lam6 is not None and Q6 is not None, "S6, lam6 and Q6 are still None."
assert np.allclose(S6, S6.T), "S6 should be symmetric."
assert max_imag < 1e-8 and ortho_ok and recon6_ok, "A symmetric matrix has real eigenvalues and an orthonormal set of eigenvectors."
print("Self-check passed.")

In [ ]:
# 6.2 (5 points): The Rayleigh quotient. For 5000 random unit vectors u (rows of U, each scaled to length 1),
# compute q(u) = u^T S6 u. Record q_max and q_min (the largest and smallest values found), and compare them with
# the largest and smallest eigenvalues. Also compute q at the top eigenvector Q6[:, -1]: q_top.

# YOUR CODE HERE
q_max = q_min = None
q_top = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.2
assert q_max is not None and q_min is not None and q_top is not None, "Fill in q_max, q_min and q_top."
assert q_max <= lam6[-1] + 1e-9 and q_min >= lam6[0] - 1e-9, "No unit vector can beat the extreme eigenvalues."
assert np.isclose(q_top, lam6[-1]), "At the top eigenvector the quotient equals the top eigenvalue."
print("Self-check passed.")

**6.3 (5 points, written response).** Explain, using the eigen-decomposition $\mathbf{S} = \mathbf{Q}\boldsymbol\Lambda\mathbf{Q}^{\mathsf T}$, why $\mathbf{u}^{\mathsf T}\mathbf{S}\mathbf{u}$ for a unit vector always lies between the smallest and largest eigenvalues, and when it reaches the maximum. (Hint: write $\mathbf{u} = \mathbf{Q}\mathbf{a}$ with $\lVert\mathbf{a}\rVert = 1$.) Lesson 7 uses this with $\mathbf{S}$ the covariance matrix of a neural population: what will "the direction $\mathbf{u}$ that maximises $\mathbf{u}^{\mathsf T}\mathbf{S}\mathbf{u}$" then mean?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Is the Timescale in the Brain or in My Smoothing? (15 points)

Part 7 of the teaching notebook fitted a $20\times20$ dynamics matrix to smoothed motor-cortex activity ([Stevenson et al., 2011](https://doi.org/10.1152/jn.00626.2010); [DREAM on CRCNS](https://crcns.org/data-sets/movements/dream); packaged by [Benjamin et al., 2018](https://github.com/KordingLab/spykesML)) and read a slowest timescale of about 0.6 s from its eigenvalues. Smoothing white noise with the same kernel already gave about 0.2 s. Here you measure how the answer depends on the amount of smoothing, and separate the part that comes from the brain from the part that comes from the method.

The cell below downloads the file (1.3 MB) the first time it runs and provides the helper functions from Part 7. Do not modify it.

In [ ]:
# PROVIDED -- do not modify.
import os, urllib.request
from scipy.io import loadmat

M1_FILE = 'm1_stevenson_2011.mat'
if not os.path.exists(M1_FILE):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/KordingLab/spykesML/ff2b1830ae07909e5f1a0811445ca3fd5732dc6f/data/' + M1_FILE, M1_FILE)
rates7 = loadmat(M1_FILE)['spikes'].T.astype(float) / 0.05     # shape (15536, 171): (time bin, neuron), spikes/s
dt7 = 0.05
train7 = np.arange(rates7.shape[0]) < rates7.shape[0] // 2

def smooth(R, sigma_bins):
    if sigma_bins == 0:
        return R.copy()
    k = np.exp(-0.5 * (np.arange(-4 * sigma_bins, 4 * sigma_bins + 1) / sigma_bins)**2); k /= k.sum()
    return np.apply_along_axis(lambda x: np.convolve(x, k, mode='same'), 0, R)

def fit_dynamics(Z, use):
    pairs = use[:-1] & use[1:]
    return np.linalg.lstsq(Z[:-1][pairs], Z[1:][pairs], rcond=None)[0].T

noise7 = np.random.default_rng(0).normal(size=(rates7.shape[0], 20))   # white noise: no dynamics at all
print("rates7:", rates7.shape)

In [ ]:
# 7.1 (4 points): Write slowest_tau(Z) that centres Z with its TRAINING-half means, fits A with fit_dynamics(Z, train7),
# and returns the slowest time constant -dt7 / ln(max |lambda|) in seconds.
# Then write prepare(sigma) that smooths rates7 with that sigma, and returns the columns of the 20 neurons with
# the largest training-half variance (after smoothing).

def slowest_tau(Z):
    # YOUR CODE HERE
    raise NotImplementedError

def prepare(sigma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1 -- tested on a simulated process with a known timescale.
x_ar = np.zeros((4000, 2)); rng_chk = np.random.default_rng(1)
for t in range(1, 4000):
    x_ar[t] = 0.9 * x_ar[t - 1] + rng_chk.normal(size=2)
tau_true = -dt7 / np.log(0.9)
saved_train = train7
train7 = np.arange(4000) < 2000
tau_est = slowest_tau(x_ar)
train7 = saved_train
assert abs(tau_est - tau_true) < 0.1 * tau_true, "slowest_tau should recover the timescale of a simple simulated process."
assert np.shape(prepare(0)) == (rates7.shape[0], 20), "prepare should return 20 columns."
print("Self-check passed.")

In [ ]:
# 7.2 (5 points): For sigma in sigmas (in bins of 50 ms), compute
#   tau_real  -- slowest_tau(prepare(sigma))
#   tau_noise -- slowest_tau(smooth(noise7, sigma))       (same smoothing, no dynamics)
#   tau_shuf  -- slowest_tau of prepare(sigma) with its TRAINING rows shuffled into a random order
#                (use np.random.default_rng(1).permutation on the training indices)
# Store each as an array with one value per sigma, and plot all three against the smoothing width in ms
# (log y-axis), with labels, a title and a legend.

sigmas = [0, 1, 2, 3, 4]

# YOUR CODE HERE
tau_real = None
tau_noise = None
tau_shuf = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert tau_real is not None and tau_noise is not None and tau_shuf is not None, "Fill in tau_real, tau_noise and tau_shuf."
assert np.shape(tau_real) == np.shape(tau_noise) == np.shape(tau_shuf) == (len(sigmas),), "One value per smoothing width."
assert np.all(np.asarray(tau_real) > 0), "Time constants are positive."
print("Self-check passed.")

In [ ]:
# 7.3 (3 points): Without any smoothing (sigma = 0), fit A to the real 20 neurons and report
#   rho_raw   -- its spectral radius
#   r2_raw    -- its one-step R^2 on the second half (compare A x_t with x_{t+1}; centre with training means)
# and the same R^2 for the trivial prediction x_{t+1} = x_t:  r2_persist_raw.

# YOUR CODE HERE
rho_raw = None
r2_raw = None
r2_persist_raw = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.3
assert rho_raw is not None and r2_raw is not None and r2_persist_raw is not None, "Fill in rho_raw, r2_raw and r2_persist_raw."
assert 0 < rho_raw < 1.5, "The spectral radius should be a positive number of order 1."
print("Self-check passed.")

**7.4 (3 points, written response — interpretation and limitation).** How does the fitted slowest timescale of real activity depend on the smoothing width, and how does it compare with smoothed noise and with shuffled data? What can you conclude about whether motor-cortex activity has slow dynamics, and what number would you be willing to report? What do the unsmoothed results in 7.3 add? Give one limitation.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — The Exponential of a Matrix (extra credit, not part of the 100 points)

The exact solution of $d\mathbf{x}/dt = \mathbf{A}\mathbf{x}$ is $\mathbf{x}(t) = e^{\mathbf{A}t}\mathbf{x}_0$, where the **matrix exponential** is $e^{\mathbf{A}t} = \mathbf{V}\operatorname{diag}(e^{\lambda_i t})\mathbf{V}^{-1}$ for a diagonalisable $\mathbf{A}$.

1. Compute $e^{\mathbf{A}_5 \cdot 0.05}$ (for $\mathbf{A}_5$ from Exercise 5) from the eigen-decomposition, and compare it with `scipy.linalg.expm`.
2. Show numerically that one step of $\mathbf{x} \leftarrow e^{\mathbf{A}\Delta t}\mathbf{x}$ is exact for any $\Delta t$, while Euler with $\Delta t = 50$ ms is unstable for this system.
3. Explain the second result using the eigenvalues of $\mathbf{I} + \Delta t\,\mathbf{A}_5$.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, and a legend where appropriate.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.